# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes (not by subscripting)
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")
print(f"Authors IDs: {[a['@id'] for a in dataset.metadata.author]}")

## 2. Data Overview
Review available record sets and fields (all entities referenced by their `@id`).

In [ ]:
# List all available record sets by their @id
record_sets = dataset.record_sets

print("Record Sets:`@id` and names:")
for rs in record_sets:
    print(f"  @id: {rs['@id']}, name: {rs.get('name', '')}")

# Choose a primary record set for exploration
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]['@id']
else:
    main_record_set_id = None

# List all fields (@id) for the primary record set
if main_record_set_id:
    print(f"\nFields in record set '@id': {main_record_set_id}")
    fields = dataset.fields(record_set=main_record_set_id)
    for field in fields:
        print(f"  Field @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType')}")

    # List columns for each field (if present)
    for field in fields:
        if 'column' in field:
            col = field['column']
            if isinstance(col, dict):
                print(f"    Column @id: {col['@id']} for field {field['@id']}")
            elif isinstance(col, list):
                for c in col:
                    print(f"    Column @id: {c['@id']} for field {field['@id']}")

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s obtained above.

In [ ]:
# Use the list of record set @id values
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded record set '@id': {rs_id} with {df.shape[0]} rows and columns:")
            print(df.columns.tolist())
            print(df.head(2))
        else:
            print(f"No records found for '@id': {rs_id}")
    except Exception as e:
        print(f"Failed to load records for '@id': {rs_id}: {e}")

# For demonstration, select the primary record set
if main_record_set_id and main_record_set_id in dataframes:
    df_main = dataframes[main_record_set_id]
    print(f"\nPrimary DataFrame Columns (@id): {df_main.columns.tolist()}")
    df_main.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping data by key attributes.

All fields and columns are referenced by their `@id`.

In [ ]:
# Automatically identify a numeric field @id from the explored fields
numeric_field_id = None
group_field_id = None

fields_main = dataset.fields(record_set=main_record_set_id)
for f in fields_main:
    if f.get('dataType') in ['Integer', 'Float', 'Number']:
        numeric_field_id = f['@id']
    if f.get('dataType') == 'Text' or f.get('dataType') == 'String':
        if not group_field_id:
            group_field_id = f['@id']

# Print chosen field ids
print(f"Numeric field @id for EDA: {numeric_field_id}")
print(f"Group field @id for EDA: {group_field_id}")

# Proceed if we have a numeric field
if numeric_field_id and numeric_field_id in df_main.columns:
    threshold = df_main[numeric_field_id].median()
    filtered_df = df_main[df_main[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Group by group_field_id if present
    if group_field_id and group_field_id in df_main.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found in main DataFrame; check field listing above.")

## 5. Visualization
Visualize data distributions or relationships between fields using record set and field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of normalized numeric field
if numeric_field_id and numeric_field_id+'_normalized' in filtered_df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id+'_normalized'], bins=20, kde=True)
    plt.title(f"Distribution of normalized field (@id): {numeric_field_id}")
    plt.xlabel(f"Normalized {numeric_field_id}")
    plt.ylabel("Count")
    plt.show()
    
# Grouped bar plot
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    plt.figure(figsize=(8,5))
    sns.barplot(data=grouped, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean of {numeric_field_id} by {group_field_id} (@id)")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean of {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook guided you through the loading, exploration, and basic analysis of the FAIR^2 clinical colorectal cancer dataset using the `mlcroissant` library, with a focus on referencing all entities by their `@id`.

- Metadata and schema attributes are accessed directly from the dataset object.
- Record sets, fields, and columns are referenced and manipulated using their unique `@id`.
- Data is filtered, normalized, grouped, and visualized using standard data science tools.
- The powerful combination of Croissant metadata and mlcroissant tools enables reproducible, transparent clinical data analysis.

For further exploration, consult the [mlcroissant documentation](https://mlcommons.org/croissant).